# Vasicek Model — Calibration, Simulation and Bond Pricing
## QRE-43

This notebook studies the Vasicek (1977) short rate model from three angles:

1. **Calibration** — fitting $\kappa$, $\theta$, $\sigma$ to the market OIS zero curve via least-squares
2. **Analytical bond pricing** — the closed-form zero-coupon bond price from the affine term structure
3. **Monte Carlo verification** — pricing ZCBs by simulation and checking convergence to the formula

**Prerequisites:** [QRE-48 — MC Rate Paths](01_mc_rate_paths.ipynb) derives the Vasicek SDE, the exact one-step transition, and antithetic variates in full. This notebook assumes that material and develops the parts specific to QRE-43.

---

**Where this fits in the system**

| Downstream notebook | What QRE-43 provides |
|---|---|
| QRE-44 (Hull-White) | Vasicek as the constant-parameter baseline; HW adds a time-dependent $\theta(t)$ to fit the curve exactly |
| QRE-45 (CIR) | Vasicek as the Gaussian comparator; CIR replaces $\sigma\,dW$ with $\sigma\sqrt{r}\,dW$ |
| QRE-49 (MC bond/swap pricing) | ZCB MC-vs-formula convergence as the benchmark for instrument pricers |
| QRE-52–57 (XVA) | Vasicek-calibrated paths as the fallback when Hull-White is unavailable |
| FRTB SA key-rate DV01 | A 1-factor model cannot independently perturb 0.25Y–30Y vertices; multi-factor extensions needed |


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats

from quant_risk.setup import base
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import VasicekProcess

np, pd, plt = base()

RNG_SEED = 42


---
## 1. The Vasicek Model

### 1.1 Stochastic Differential Equation

$$dr(t) = \kappa\bigl(\theta - r(t)\bigr)\,dt + \sigma\,dW(t)$$

| Symbol | Meaning | Units | Typical EUR (2026) |
|---|---|---|---|
| $\kappa > 0$ | Mean reversion speed | year$^{-1}$ | 0.05 – 0.30 |
| $\theta$ | Long-run mean rate | % | 2.0 – 3.5 |
| $\sigma > 0$ | Instantaneous vol | %/$\sqrt{\text{year}}$ | 0.3 – 1.0 |

The drift $\kappa(\theta - r)$ is a linear **restoring force**: the further $r$ is from $\theta$, the stronger the pull back. The mean-reversion half-life is $\ln 2 / \kappa$ — at $\kappa = 0.1$ that is roughly 7 years, consistent with the slow drift of policy rate expectations.

### 1.2 Analytical Solution

Multiplying both sides by the integrating factor $e^{\kappa t}$ and integrating:

$$r(t) = r(0)\,e^{-\kappa t} + \theta\bigl(1 - e^{-\kappa t}\bigr) + \sigma \int_0^t e^{-\kappa(t-s)}\,dW(s)$$

The three terms are: (1) the decaying memory of the starting rate, (2) the long-run mean growing in as the memory fades, and (3) a Gaussian stochastic integral with mean 0 and variance $\frac{\sigma^2}{2\kappa}(1 - e^{-2\kappa t})$.

### 1.3 Distributional Properties

$r(t)$ is **normally distributed** for all $t$:

$$r(t) \sim \mathcal{N}\!\left(\mu(t),\; v^2(t)\right)$$

$$\mu(t) = r(0)\,e^{-\kappa t} + \theta(1 - e^{-\kappa t}) \qquad v^2(t) = \frac{\sigma^2}{2\kappa}\bigl(1 - e^{-2\kappa t}\bigr)$$

As $t \to \infty$: $\mu \to \theta$, $v^2 \to \sigma^2/(2\kappa)$ — the **stationary distribution** $\mathcal{N}(\theta, \sigma^2/(2\kappa))$.

### 1.4 One-Step Exact Simulation

The one-step conditional distribution follows directly from the analytical solution:

$$r(t+\Delta t)\,\big|\,r(t) \sim \mathcal{N}\!\left(r(t)\,e^{-\kappa\Delta t} + \theta(1-e^{-\kappa\Delta t}),\;\frac{\sigma^2}{2\kappa}(1-e^{-2\kappa\Delta t})\right)$$

This is **exact** — no Euler approximation, no dependence on step size. The production class `VasicekProcess` uses this formula. See QRE-48 for a step-by-step derivation and comparison against Euler-Maruyama.

### 1.5 Negative Rates

Because $r(t)$ is Gaussian, it can go negative with probability:

$$\Pr(r(t) < 0) = \Phi\!\left(\frac{-\mu(t)}{v(t)}\right)$$

In the stationary distribution: $\Pr(r < 0) = \Phi(-\theta\sqrt{2\kappa}/\sigma)$.
For $\theta = 2.5\%$, $\kappa = 0.1$, $\sigma = 0.5\%$: $\Pr(r < 0) \approx 0.08\%$ — negligible.
For $\sigma = 1.5\%$ the same formula gives $\approx 7\%$ — material for vol-stressed scenarios.
CIR (QRE-45) enforces non-negativity structurally via a square-root diffusion term.


In [ ]:
# ── Parameter sensitivity: how κ, θ, σ shape the term structure of rates ────
# The drift field dr/dt = κ(θ - r) is a straight line in (r, drift) space.
# The slope is -κ (steeper → faster reversion) and the zero-crossing is θ.

r_grid = np.linspace(-0.5, 6.0, 300)  # rate values in percent

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# ── Left: drift field for different κ ────────────────────────────────────────
ax = axes[0]
theta_ref, sigma_ref = 2.5, 0.5
for kappa_val, color in [(0.05, 'steelblue'), (0.15, 'darkorange'), (0.40, 'forestgreen')]:
    drift = kappa_val * (theta_ref - r_grid)
    hl    = np.log(2) / kappa_val
    ax.plot(r_grid, drift, color=color, lw=1.8,
            label=f'κ={kappa_val:.2f}  (half-life {hl:.1f}y)')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(theta_ref, color='black', lw=0.8, ls=':', alpha=0.5)
ax.set_xlabel('r(t)  (%)')
ax.set_ylabel('drift = κ(θ − r)  (%/year)')
ax.set_title('Drift field for varying κ  (θ=2.5%, σ=0.5%)')
ax.legend(fontsize=7)

# ── Middle: stationary distribution for different σ ───────────────────────────
ax = axes[1]
kappa_ref = 0.10
for sigma_val, color in [(0.30, 'steelblue'), (0.60, 'darkorange'), (1.20, 'firebrick')]:
    stat_std  = sigma_val / np.sqrt(2 * kappa_ref)
    prob_neg  = stats.norm.cdf(0, theta_ref, stat_std) * 100
    ax.plot(r_grid, stats.norm.pdf(r_grid, theta_ref, stat_std),
            color=color, lw=1.8,
            label=f'σ={sigma_val:.2f}%  Pr(r<0)={prob_neg:.2f}%')
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.6)
ax.set_xlabel('r  (%)')
ax.set_ylabel('Density')
ax.set_title(f'Stationary distribution N(θ, σ²/2κ)
(κ={kappa_ref}, θ={theta_ref}%)')
ax.legend(fontsize=7)

# ── Right: mean and ±1σ band over time for different r0 ──────────────────────
ax = axes[2]
kappa_ref, sigma_ref = 0.10, 0.50
t_plot = np.linspace(0, 15, 300)
for r0_val, color in [(0.5, 'steelblue'), (2.5, 'darkorange'), (5.0, 'firebrick')]:
    mu_t    = r0_val * np.exp(-kappa_ref * t_plot) + theta_ref * (1 - np.exp(-kappa_ref * t_plot))
    sigma_t = sigma_ref * np.sqrt((1 - np.exp(-2 * kappa_ref * t_plot)) / (2 * kappa_ref))
    ax.plot(t_plot, mu_t, color=color, lw=1.8, label=f'r(0)={r0_val:.1f}%')
    ax.fill_between(t_plot, mu_t - sigma_t, mu_t + sigma_t, color=color, alpha=0.12)
ax.axhline(theta_ref, color='black', lw=0.8, ls=':', alpha=0.6, label=f'θ={theta_ref}%')
ax.set_xlabel('t  (years)')
ax.set_ylabel('E[r(t)] ± σ(t)  (%)')
ax.set_title('Mean reversion from different starting rates
(κ=0.10, θ=2.5%, σ=0.50%)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# Stationary stats for the reference parameterisation
kappa, theta, sigma = 0.10, 2.5, 0.50
stat_std = sigma / np.sqrt(2 * kappa)
print(f"Reference: κ={kappa}, θ={theta}%, σ={sigma}%/√yr")
print(f"  Half-life of mean reversion: {np.log(2)/kappa:.1f} years")
print(f"  Stationary std dev:          {stat_std:.4f}%")
print(f"  Pr(r < 0) at stationarity:   {stats.norm.cdf(0, theta, stat_std)*100:.4f}%")


---
## 2. Analytical Bond Pricing — Affine Term Structure

### 2.1 What Is an Affine Term Structure Model?

A short rate model has an **affine term structure** if zero-coupon bond prices take the form:

$$P(0, T) = \exp\bigl(\alpha(T) - \beta(T)\cdot r(0)\bigr)$$

where $\alpha(T)$ and $\beta(T)$ are deterministic functions of maturity, independent of $r(0)$. Vasicek is the canonical example; Hull-White and CIR are also affine. The practical payoff is enormous: once you have $\alpha$ and $\beta$, you can price a whole term structure of zero-coupon bonds in microseconds.

### 2.2 Deriving $\beta(T)$ and $\alpha(T)$

Under the risk-neutral measure $\mathbb{Q}$, the bond price satisfies the **bond pricing PDE**:

$$\frac{\partial P}{\partial t} + \kappa(\theta - r)\frac{\partial P}{\partial r} + \frac{\sigma^2}{2}\frac{\partial^2 P}{\partial r^2} - rP = 0, \qquad P(T,T) = 1$$

Substituting the affine ansatz $P = \exp(\alpha(\tau) - \beta(\tau)\cdot r)$ where $\tau = T - t$ and separating constant and $r$-proportional terms:

**$r$-coefficient** (must vanish):
$$\beta'(\tau) = 1 - \kappa\,\beta(\tau), \qquad \beta(0) = 0$$
$$\Rightarrow\quad \boxed{\beta(\tau) = \frac{1 - e^{-\kappa\tau}}{\kappa}}$$

**Constant term** (ODE for $\alpha$):
$$\alpha'(\tau) = \kappa\theta\,\beta(\tau) - \frac{\sigma^2}{2}\beta(\tau)^2, \qquad \alpha(0) = 0$$

Integrating and using $\int_0^\tau \beta(s)\,ds = (\tau - \beta(\tau))/\kappa$ and $\int_0^\tau \beta(s)^2\,ds = (\tau - 2\beta(\tau) + \beta(2\tau/2)\cdot\ldots)/\ldots$:

$$\boxed{\alpha(\tau) = \left(\theta - \frac{\sigma^2}{2\kappa^2}\right)(\beta(\tau) - \tau) - \frac{\sigma^2}{4\kappa}\beta(\tau)^2}$$

### 2.3 Zero-Coupon Bond Price and Yield Curve

The zero-coupon bond price and continuously compounded yield are:

$$P(0,T) = \exp\!\left(\alpha(T) - \beta(T)\cdot r(0)\right)$$

$$R(0,T) = -\frac{\ln P(0,T)}{T} = -\frac{\alpha(T)}{T} + \frac{\beta(T)}{T}\cdot r(0)$$

**Key structural insight:** $R(0,T)$ is a linear function of $r(0)$. A $+1\%$ parallel shift in $r(0)$ shifts the entire yield curve by $\beta(T)/T$ — which is less than 1 at all maturities and converges to $1/\kappa$ as $T \to \infty$. The Vasicek yield curve can only produce **normal, inverted, and mildly humped** shapes; it cannot fit arbitrary curvature.

### 2.4 Long-End Yield

As $T \to \infty$: $\beta(T)/T \to 0$ and $-\alpha(T)/T \to \theta - \sigma^2/(2\kappa^2) \equiv R_\infty$.

$$R_\infty = \theta - \frac{\sigma^2}{2\kappa^2}$$

This is the **long-run yield implied by the model**. Note it is below $\theta$: the volatility term $\sigma^2/(2\kappa^2)$ is a Jensen convexity correction arising because $P = E^{\mathbb{Q}}[e^{-\int r\,ds}]$ is a convex function of the rate path.


In [ ]:
# ── Vasicek affine bond pricing ───────────────────────────────────────────────
# These three functions implement B(T), alpha(T) and the resulting ZCB price
# and yield. They are vectorised over T so a full yield curve is one call.

def vasicek_B(T: np.ndarray, kappa: float) -> np.ndarray:
    """β(T) = (1 - exp(-κT)) / κ.  Limit as T→0 is 1 (L'Hôpital)."""
    # Avoid division by zero at T=0 by returning 0 there
    out = np.where(T > 0, (1 - np.exp(-kappa * T)) / kappa, 0.0)
    return out


def vasicek_alpha(T: np.ndarray, kappa: float, theta: float, sigma: float) -> np.ndarray:
    """α(T) in the affine formula P = exp(α - β·r₀/100).
    theta and sigma are in percent; converted to decimal inside."""
    B       = vasicek_B(T, kappa)
    theta_d = theta / 100          # percent → decimal
    sigma_d = sigma / 100
    R_inf_d = theta_d - sigma_d ** 2 / (2 * kappa ** 2)
    return R_inf_d * (B - T) - sigma_d ** 2 / (4 * kappa) * B ** 2


def vasicek_zcb(T: np.ndarray, r0: float,
                kappa: float, theta: float, sigma: float) -> np.ndarray:
    """P(0,T) = exp(α(T) - β(T)·r₀/100). r0 in percent; /100 converts to decimal."""
    return np.exp(vasicek_alpha(T, kappa, theta, sigma) - vasicek_B(T, kappa) * r0 / 100)


def vasicek_yield(T: np.ndarray, r0: float,
                  kappa: float, theta: float, sigma: float) -> np.ndarray:
    """Continuously compounded zero yield R(0,T) in percent."""
    P = vasicek_zcb(T, r0, kappa, theta, sigma)
    return -np.log(P) / T * 100   # ×100: decimal → percent


# ── Demo: yield curves under different parameter sets ────────────────────────
T_grid = np.linspace(0.25, 30, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Left: effect of r0 (level shift) — same κ, θ, σ, different starting rate
ax = axes[0]
kappa_d, theta_d, sigma_d = 0.10, 2.5, 0.50
# R_inf in percent: convert θ,σ to decimal inside formula, then ×100
def R_inf_pct(kappa, theta, sigma):
    theta_dd, sigma_dd = theta/100, sigma/100
    return (theta_dd - sigma_dd**2/(2*kappa**2)) * 100
for r0_val, ls, color in [(1.0, '-', 'steelblue'), (2.5, '-', 'darkorange'),
                           (4.5, '-', 'firebrick'), (0.0, '--', 'purple')]:
    ax.plot(T_grid, vasicek_yield(T_grid, r0_val, kappa_d, theta_d, sigma_d),
            ls=ls, color=color, lw=1.5,
            label=f'r(0)={r0_val:.1f}%')
R_inf_ref = R_inf_pct(kappa_d, theta_d, sigma_d)
ax.axhline(R_inf_ref, lw=0.8, ls=':', color='black',
           alpha=0.5, label=f'R∞={R_inf_ref:.2f}%')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield R(0,T) (%)')
ax.set_title('Effect of r(0) — κ=0.10, θ=2.5%, σ=0.50%')
ax.legend(fontsize=7)

# Middle: effect of σ (convexity / long-end suppression)
ax = axes[1]
r0_d = 2.5
for sigma_val, color in [(0.20, 'steelblue'), (0.60, 'darkorange'), (1.20, 'firebrick')]:
    ax.plot(T_grid, vasicek_yield(T_grid, r0_d, kappa_d, theta_d, sigma_val),
            color=color, lw=1.5,
            label=f'σ={sigma_val:.2f}%  R∞={R_inf_pct(kappa_d, theta_d, sigma_val):.2f}%')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield R(0,T) (%)')
ax.set_title('Effect of σ on long-end yield
(κ=0.10, θ=2.5%, r(0)=2.5%)')
ax.legend(fontsize=7)

# Right: effect of κ (speed at which curve converges to R∞)
ax = axes[2]
for kappa_val, color in [(0.03, 'steelblue'), (0.10, 'darkorange'), (0.40, 'firebrick')]:
    hl = np.log(2) / kappa_val
    ax.plot(T_grid, vasicek_yield(T_grid, r0_d, kappa_val, theta_d, sigma_d),
            color=color, lw=1.5,
            label=f'κ={kappa_val:.2f} (HL {hl:.1f}y)  R∞={R_inf_pct(kappa_val, theta_d, sigma_d):.2f}%')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield R(0,T) (%)')
ax.set_title('Effect of κ on convergence speed
(θ=2.5%, σ=0.50%, r(0)=2.5%)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

# Quick sanity check: at T→0, yield should approach r0
T_short = np.array([0.001])
print(f"R(0, T→0) = {vasicek_yield(T_short, 2.5, 0.10, 2.5, 0.50)[0]:.4f}%  (should approach r0=2.5%)")
R_inf_check = R_inf_pct(0.10, 2.5, 0.50)
print(f"R_∞       = (θ_dec - σ_dec²/2κ²) × 100 = {R_inf_check:.4f}%")


---
## 3. Calibration to the OIS Term Structure

### 3.1 The Calibration Problem

Given market zero yields $\{(T_i,\, R_i^{\text{OIS}})\}_{i=1}^{n}$ bootstrapped from the OIS curve, find the Vasicek parameters $(\kappa, \theta, \sigma)$ and initial short rate $r_0$ such that:

$$\hat{\kappa},\,\hat{\theta},\,\hat{\sigma} = \arg\min_{\kappa,\theta,\sigma} \sum_{i=1}^{n} w_i \bigl(R^{\text{model}}(T_i;\,\kappa,\theta,\sigma,r_0) - R_i^{\text{OIS}}\bigr)^2$$

We fix $r_0 = f^M(0, 1\text{M})$ — the market 1-month forward rate — to anchor the short end, and optimise over $(\kappa, \theta, \sigma)$ only.

### 3.2 The 1-Factor Limitation

The Vasicek model has only three free parameters after fixing $r_0$. The yield curve implied by the model is:

$$R(0,T) = R_\infty + \underbrace{\frac{\beta(T)}{T}}_{{\leq 1/\kappa}}(r_0 - R_\infty) \qquad \text{with } R_\infty = \theta - \frac{\sigma^2}{2\kappa^2}$$

This family of curves has **level** (controlled by $R_\infty$) and **slope** (controlled by $r_0 - R_\infty$ and the speed $\kappa$). A market OIS curve with genuine curvature (e.g., a belly hump from policy rate expectations) cannot be fit exactly — the calibration will show residuals at intermediate maturities.

**Hull-White** removes this limitation by replacing $\theta$ with the time-dependent function $\theta(t)$ calibrated to reproduce every market pillar exactly (QRE-44).

### 3.3 Weighting

We apply maturity-proportional weights $w_i = T_i / \sum_j T_j$ so that long-maturity points (which carry more duration risk) are not dominated by the many short-end pillars. For CVA/FVA use cases, the 2Y–10Y region is the most important and benefits from this weighting.


In [ ]:
# ── Load OIS curve and extract zero yields ────────────────────────────────────
from quant_risk.config import PROCESSED_DIR

ois = OISCurve.from_processed(str(PROCESSED_DIR))
print(ois.describe())

# Standard calibration pillars — same as FRTB SA key-rate vertices plus 2Y, 3Y
cal_maturities = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0])

# Extract OIS zero yields at each pillar
ois_yields = np.array([ois.zero_rate(T) for T in cal_maturities])

# Anchor: 1-month instantaneous forward rate from the OIS curve
# Use a 1M window rather than instantaneous_forward() to avoid OISCurve
# integer-date arithmetic issues at sub-monthly tenors (see HW._theta() docs).
r0_market = ois.forward_rate(1/12, 2/12)
print(f"\nCalibration pillars:")
for T, R in zip(cal_maturities, ois_yields):
    print(f"  {T:5.2f}Y : {R:.4f}%")
print(f"\nAnchor short rate r0 = {r0_market:.4f}%  (1M→2M OIS forward)")


In [ ]:
# ── Calibration objective and optimisation ────────────────────────────────────

# Maturity-proportional weights: longer tenors carry more rate sensitivity
weights = cal_maturities / cal_maturities.sum()


def calibration_objective(params: np.ndarray) -> float:
    """Sum of maturity-weighted squared yield errors."""
    kappa_c, theta_c, sigma_c = params

    # Reject economically implausible parameters during the optimisation
    if kappa_c <= 0 or sigma_c <= 0:
        return 1e10

    model_yields = vasicek_yield(cal_maturities, r0_market,
                                  kappa_c, theta_c, sigma_c)

    # Weighted SSE in basis-point-squared units (×10000 for numerical scaling)
    sse = np.sum(weights * (model_yields - ois_yields) ** 2) * 1e4
    return sse


# Initial guess: κ=0.1 (7Y half-life), θ≈long-end yield, σ=0.5% (moderate vol)
# Multiple restarts reduce the risk of local minima in this non-convex surface
theta_init  = ois_yields[-1]   # long-end yield as θ starting point
x0_grid     = [(0.05, theta_init, 0.4), (0.15, theta_init, 0.6), (0.30, theta_init, 0.8)]
best_result = None

for x0_guess in x0_grid:
    result = minimize(
        calibration_objective,
        x0     = np.array(x0_guess),
        method = 'Nelder-Mead',
        options = {'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-10},
    )
    if best_result is None or result.fun < best_result.fun:
        best_result = result

kappa_cal, theta_cal, sigma_cal = best_result.x
R_inf_cal = R_inf_pct(kappa_cal, theta_cal, sigma_cal)
feller_ok = 2 * kappa_cal * theta_cal > sigma_cal**2   # Vasicek is always Gaussian, but informative

print("Calibrated Vasicek parameters")
print(f"  κ     = {kappa_cal:.4f}  (half-life {np.log(2)/kappa_cal:.1f}y)")
print(f"  θ     = {theta_cal:.4f}%")
print(f"  σ     = {sigma_cal:.4f}%/√yr")
print(f"  R∞    = θ − σ²/2κ² = {R_inf_cal:.4f}%")
print(f"  r(0)  = {r0_market:.4f}%  (fixed from OIS 1M forward)")
print(f"  Feller (2κθ > σ²): {2*kappa_cal*theta_cal:.4f} vs {sigma_cal**2:.4f}  → {'satisfied' if feller_ok else 'violated'}")

# ── Residuals table ───────────────────────────────────────────────────────────
model_yields_cal = vasicek_yield(cal_maturities, r0_market, kappa_cal, theta_cal, sigma_cal)
residuals_bps    = (model_yields_cal - ois_yields) * 100  # in basis points

print("\nCalibration residuals (model − OIS)")
print(f"  {'Maturity':>8}  {'OIS':>8}  {'Model':>8}  {'Error (bps)':>12}")
for T, R_ois, R_mod, err in zip(cal_maturities, ois_yields, model_yields_cal, residuals_bps):
    flag = '  ← large' if abs(err) > 5 else ''
    print(f"  {T:>8.2f}Y  {R_ois:>8.4f}%  {R_mod:>8.4f}%  {err:>+12.2f} bps{flag}")

rmse_bps = np.sqrt(np.mean(residuals_bps**2))
print(f"\nRMSE = {rmse_bps:.2f} bps")


In [ ]:
# ── Calibration quality: model vs OIS term structure ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

T_dense = np.linspace(0.1, 30, 500)
model_dense = vasicek_yield(T_dense, r0_market, kappa_cal, theta_cal, sigma_cal)

# Left: yield curves
ax = axes[0]
ax.plot(T_dense, model_dense, '-', color='steelblue', lw=2.0, label='Vasicek (calibrated)')
ax.plot(cal_maturities, ois_yields, 'o', color='black', ms=6, zorder=5, label='OIS pillars')
ax.axhline(R_inf_cal, lw=0.9, ls=':', color='steelblue', alpha=0.6,
           label=f'R∞ = {R_inf_cal:.2f}%')
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Zero yield (%)')
ax.set_title('Vasicek fit to OIS zero curve')
ax.legend()

# Right: residuals by maturity
ax = axes[1]
colors = ['firebrick' if abs(e) > 5 else 'steelblue' for e in residuals_bps]
ax.bar(cal_maturities, residuals_bps, width=0.6, color=colors, alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(5, color='firebrick', lw=0.8, ls='--', alpha=0.5)
ax.axhline(-5, color='firebrick', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Maturity T (years)')
ax.set_ylabel('Residual (bps)')
ax.set_title(f'Calibration residuals — RMSE = {rmse_bps:.2f} bps\n'
             f'(dashed line = 5 bps threshold)')

plt.tight_layout()
plt.show()

print("Insight: residuals at intermediate maturities (2Y–10Y) reflect the")
print("1-factor limitation — Vasicek cannot independently fit all curvature modes.")
print("Hull-White resolves this by using a time-dependent θ(t) (QRE-44).")


---
## 4. Simulation with Calibrated Parameters

With calibrated $(\hat{\kappa}, \hat{\theta}, \hat{\sigma})$ and $r_0$ anchored to the market short rate, we simulate rate paths. The exact one-step conditional distribution (derived in QRE-48) is used throughout — no Euler approximation.

The fan chart below shows the theoretical $\mu(t) \pm$ percentile bands, which should contain the simulated paths in the expected proportions. This is a useful sanity check that the production class `VasicekProcess` correctly propagates the calibrated parameters.


In [ ]:
# ── Simulate paths with calibrated parameters ─────────────────────────────────
vasicek_proc = VasicekProcess(kappa=kappa_cal, theta=theta_cal, sigma=sigma_cal)
print(vasicek_proc.describe())

T_sim    = 10.0   # 10-year horizon — covers a full rate cycle
n_steps  = 120    # monthly steps (standard for XVA simulation)
n_paths  = 2000
t_grid   = np.linspace(0, T_sim, n_steps + 1)

paths = vasicek_proc.simulate(
    x0        = r0_market,  # calibrated initial rate
    T         = T_sim,
    n_steps   = n_steps,
    n_paths   = n_paths,
    antithetic= True,       # halves variance; n_paths must be even
    seed      = RNG_SEED,
)

# ── Theoretical mean and percentile bands ────────────────────────────────────
mu_t     = r0_market * np.exp(-kappa_cal * t_grid) + theta_cal * (1 - np.exp(-kappa_cal * t_grid))
sigma_t  = sigma_cal * np.sqrt((1 - np.exp(-2 * kappa_cal * t_grid)) / (2 * kappa_cal))

# Theoretical quantiles: 5th, 25th, 75th, 95th
q_05 = stats.norm.ppf(0.05, mu_t, sigma_t)
q_25 = stats.norm.ppf(0.25, mu_t, sigma_t)
q_75 = stats.norm.ppf(0.75, mu_t, sigma_t)
q_95 = stats.norm.ppf(0.95, mu_t, sigma_t)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: fan chart
ax = axes[0]
# Draw 30 individual paths for visual texture
for i in range(30):
    ax.plot(t_grid, paths[i], lw=0.4, color='steelblue', alpha=0.3)
ax.fill_between(t_grid, q_05, q_95, color='steelblue', alpha=0.12, label='5th–95th pctile (theory)')
ax.fill_between(t_grid, q_25, q_75, color='steelblue', alpha=0.22, label='25th–75th pctile (theory)')
ax.plot(t_grid, mu_t, '--', color='black', lw=1.5, label='Theoretical E[r(t)]')
ax.plot(t_grid, paths.mean(axis=0), '-', color='darkorange', lw=1.2, label='Simulated mean')
ax.axhline(theta_cal, lw=0.8, ls=':', color='gray', alpha=0.7, label=f'θ={theta_cal:.2f}%')
ax.set_xlabel('t (years)')
ax.set_ylabel('r(t) (%)')
ax.set_title(f'Vasicek fan chart — κ={kappa_cal:.3f}, θ={theta_cal:.3f}%, σ={sigma_cal:.3f}%')
ax.legend(fontsize=7)

# Right: terminal distribution — simulated vs theoretical N(μ(T), v²(T))
ax = axes[1]
terminal_rates = paths[:, -1]
r_range = np.linspace(terminal_rates.min() - 0.3, terminal_rates.max() + 0.3, 300)
ax.hist(terminal_rates, bins=60, density=True, color='steelblue', alpha=0.6, label='Simulated r(T)')
ax.plot(r_range, stats.norm.pdf(r_range, mu_t[-1], sigma_t[-1]),
        '-', color='black', lw=2.0, label=f'N(μ={mu_t[-1]:.2f}%, σ={sigma_t[-1]:.2f}%)')
ax.set_xlabel(f'r({T_sim:.0f}Y) (%)')
ax.set_ylabel('Density')
ax.set_title(f'Terminal distribution r({T_sim:.0f}Y) — N={n_paths:,} paths')
ax.legend()

plt.tight_layout()
plt.show()

# Empirical vs theoretical check
print(f"Terminal rate statistics (T={T_sim}Y, {n_paths:,} paths)")
print(f"  Simulated:   mean={terminal_rates.mean():.4f}%  std={terminal_rates.std():.4f}%")
print(f"  Theoretical: mean={mu_t[-1]:.4f}%  std={sigma_t[-1]:.4f}%")
print(f"  Negative rates in simulation: {(terminal_rates < 0).sum()} / {n_paths}")


---
## 5. Monte Carlo Zero-Coupon Bond Pricing vs. Analytical Formula

### 5.1 Risk-Neutral Pricing

Under the risk-neutral measure $\mathbb{Q}$, a zero-coupon bond with maturity $T$ is worth the discounted expectation of $1$:

$$P^{\text{MC}}(0, T) = \mathbb{E}^{\mathbb{Q}}\!\left[e^{-\int_0^T r(s)\,ds}\right]$$

This is the **stochastic discount factor** (SDF) averaged across paths. It is exactly what `MCSimulator.price(lambda paths: np.ones(n_paths))` computes. The path integral $\int_0^T r(s)\,ds$ is approximated by the Riemann sum $\sum_{i=0}^{N-1} r(t_i)\,\Delta t$, consistent with the pre-computed `_cum_int` in `MCSimulator`.

### 5.2 Why Compare?

The analytical formula $P^{\text{Vasicek}}(0,T) = \exp(\alpha(T) - \beta(T)\cdot r_0)$ gives the exact value that the MC estimator is converging to. The comparison serves three purposes:

1. **Correctness check** — a large gap signals a bug in either the path simulation or the integral approximation
2. **Convergence analysis** — verifies that the MC standard error shrinks as $O(1/\sqrt{N})$
3. **Calibration consistency** — if the analytical price matches the OIS discount factor, the calibration is self-consistent

### 5.3 Discretisation Bias

Even with exact path simulation (exact conditional normal draws), the path integral is still approximated. The bias is $O(\Delta t)$ — halving the number of steps doubles the time per path but halves the bias. For XVA (where the objective is an exposure profile, not a single price), monthly steps ($\Delta t = 1/12$) are standard and the bias is negligible relative to the statistical uncertainty from path count.


In [ ]:
# ── MC ZCB pricing vs Vasicek analytical formula ─────────────────────────────
# Price ZCBs at several maturities using MC and compare to the closed form.

pricing_maturities = np.array([1.0, 2.0, 5.0, 10.0])

# Analytical prices from the calibrated Vasicek formula
P_analytic = vasicek_zcb(pricing_maturities, r0_market, kappa_cal, theta_cal, sigma_cal)
# OIS market discount factors — the ground truth
P_ois      = np.array([ois.discount_factor(T) for T in pricing_maturities])

print("ZCB prices: Vasicek analytical vs OIS market discount factors")
print(f"  {'T':>5}  {'P_Vasicek':>12}  {'P_OIS':>12}  {'Diff (bps)':>12}")
for T, Pv, Po in zip(pricing_maturities, P_analytic, P_ois):
    diff_bps = (Pv - Po) * 10000
    print(f"  {T:>5.1f}Y  {Pv:>12.6f}  {Po:>12.6f}  {diff_bps:>+12.2f}")

print("\nNote: the Vasicek model cannot exactly fit all OIS pillars (1-factor limit).")
print("The difference above equals the calibration residual expressed as a price error.")


In [ ]:
# ── MC convergence analysis: SE vs path count ─────────────────────────────────
# For each maturity, run MC with increasing path counts and measure the
# standard error of the ZCB price estimate.

T_mc      = 5.0                       # price the 5Y ZCB
n_steps_mc = 60                       # monthly steps
P_true    = vasicek_zcb(np.array([T_mc]), r0_market, kappa_cal, theta_cal, sigma_cal)[0]

path_counts = np.array([100, 250, 500, 1000, 2000, 5000, 10000])
n_trials    = 200    # independent price estimates per path count

se_plain = np.zeros(len(path_counts))
se_anti  = np.zeros(len(path_counts))
mean_mc  = np.zeros(len(path_counts))

for j, n in enumerate(path_counts):
    est_plain = np.zeros(n_trials)
    est_anti  = np.zeros(n_trials)

    for trial in range(n_trials):
        # Plain MC
        proc_t = VasicekProcess(kappa=kappa_cal, theta=theta_cal, sigma=sigma_cal)
        p_plain = proc_t.simulate(x0=r0_market, T=T_mc, n_steps=n_steps_mc,
                                   n_paths=n, antithetic=False, seed=trial)
        # Path integral ∫r dt via Riemann sum; divide by 100 (percent → decimal)
        dt_mc = T_mc / n_steps_mc
        cum_int_plain = np.cumsum(p_plain[:, :-1] / 100.0 * dt_mc, axis=1)
        sdf_plain     = np.exp(-cum_int_plain[:, -1])
        est_plain[trial] = sdf_plain.mean()   # E[D(0,T)] = P(0,T)

        # Antithetic MC (n paths: n/2 base + n/2 antithetic)
        p_anti = proc_t.simulate(x0=r0_market, T=T_mc, n_steps=n_steps_mc,
                                   n_paths=n, antithetic=True, seed=trial)
        cum_int_anti = np.cumsum(p_anti[:, :-1] / 100.0 * dt_mc, axis=1)
        sdf_anti     = np.exp(-cum_int_anti[:, -1])
        est_anti[trial] = sdf_anti.mean()

    se_plain[j] = est_plain.std()
    se_anti[j]  = est_anti.std()
    mean_mc[j]  = est_anti.mean()    # antithetic estimate at full path count

theo_se = se_plain[0] * np.sqrt(path_counts[0] / path_counts)   # O(1/√N) reference

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: convergence of MC price to analytical
ax = axes[0]
ax.semilogx(path_counts, [mean_mc[j] for j in range(len(path_counts))],
            'o-', color='steelblue', lw=1.5, ms=6, label='MC price (antithetic)')
ax.axhline(P_true, color='black', lw=1.2, ls='--', label=f'Analytical {P_true:.6f}')
ax.axhline(P_ois[2], color='darkorange', lw=1.2, ls=':', label=f'OIS market {P_ois[2]:.6f}')
ax.set_xlabel('Number of paths')
ax.set_ylabel(f'P(0, {T_mc:.0f}Y)')
ax.set_title(f'MC ZCB price convergence — T={T_mc:.0f}Y')
ax.legend()

# Right: SE vs N — both methods on log-log scale
ax = axes[1]
ax.loglog(path_counts, se_plain, 'o-', color='steelblue',  lw=1.5, ms=5, label='Plain MC')
ax.loglog(path_counts, se_anti,  's-', color='darkorange', lw=1.5, ms=5, label='Antithetic')
ax.loglog(path_counts, theo_se,  '--', color='gray',       lw=1.2, alpha=0.7,
          label=r'$O(1/\sqrt{N})$')
ax.set_xlabel('Number of paths N')
ax.set_ylabel(f'SE of P(0, {T_mc:.0f}Y)')
ax.set_title('Standard error vs path count')
ax.legend()

plt.tight_layout()
plt.show()

bias = mean_mc[-1] - P_true
var_reduction = (se_plain[-1] / se_anti[-1]) ** 2
print(f"At N={path_counts[-1]:,} paths:")
print(f"  MC price (antithetic): {mean_mc[-1]:.6f}")
print(f"  Analytical (Vasicek):  {P_true:.6f}")
print(f"  Discretisation bias:   {bias:+.6f}  ({bias/(P_true)*1e4:+.2f} bps)")
print(f"  Antithetic speedup:    {var_reduction:.1f}× (equivalent plain paths for same SE)")


---
## 6. Limitations and What Comes Next

### 6.1 What Vasicek Cannot Do

| Limitation | Consequence | Resolution |
|---|---|---|
| Gaussian rates — can go negative | Mildly unrealistic at very high σ; was acceptable during EUR NIRP 2014–2022 | CIR (QRE-45) enforces $r \geq 0$ via $\sqrt{r}$ diffusion |
| 1-factor — only parallel shifts | Cannot independently shock 0.25Y vs 10Y → FRTB key-rate DV01 misspecified | Multi-factor models (2F HJM, LMM) |
| Constant $\theta$ — flat long end | Cannot fit an OIS curve with curvature exactly | Hull-White (QRE-44) replaces $\theta$ with $\theta(t)$ |
| Constant $\kappa, \sigma$ — no time dependence | Implied vol structure is flat; swaption surface not reproducible | Hull-White with time-varying $\sigma(t)$ |

### 6.2 When Vasicek Is Appropriate

- **Calibration benchmarks** — the 3-parameter form makes it easy to interpret parameter stability over time
- **Analytical pricing** — the affine formula gives instant ZCB prices for scenario analysis and stress testing
- **Pedagogical use** — the clean Gaussian structure allows exact derivations and closed-form checks for MC code
- **Low-dimensional XVA** — when the priority is exposure profile shape rather than precise swaption calibration

### 6.3 Production Class

The `VasicekProcess` in `src/quant_risk/models/rates.py` (QRE-46) implements exact simulation with pre-computed constants $e^{-\kappa \Delta t}$ and $v_{\Delta t}$, antithetic variates via `StochasticProcess._draw_normals()`, and full parameter validation. `MCSimulator` (QRE-51) wraps it to provide SDF computation, exposure profiles, and ZCB pricing via a payoff callable — exactly the pattern demonstrated in Section 5 above.


In [ ]:
# ── Production class: VasicekProcess + MCSimulator ────────────────────────────
from quant_risk.models.simulator import MCSimulator

# Build the simulator with calibrated parameters
sim = MCSimulator(
    process   = VasicekProcess(kappa=kappa_cal, theta=theta_cal, sigma=sigma_cal),
    x0        = r0_market,
    T         = 10.0,
    n_steps   = 120,       # monthly
    n_paths   = 2000,
    antithetic= True,
    seed      = RNG_SEED,
)
print(sim.describe())

# ZCB prices via MCSimulator.price() — one line per maturity
print("\nZCB prices via MCSimulator.price()")
for T_p in [1.0, 2.0, 5.0, 10.0]:
    # payoff_fn: always pay 1 at maturity T_p
    # MCSimulator.price() handles discounting D(0,T_p) internally
    P_mc  = sim.price(lambda paths: np.ones(paths.shape[0]))
    P_ana = vasicek_zcb(np.array([T_p]), r0_market, kappa_cal, theta_cal, sigma_cal)[0]
    # Note: price() discounts to the full horizon T=10Y, not to T_p.
    # For a maturity-specific ZCB we need sdf(T_p) directly:
    P_mc_t = sim.sdf(T_p).mean()
    print(f"  T={T_p:5.1f}Y  MC={P_mc_t:.6f}  Analytical={P_ana:.6f}  "
          f"diff={abs(P_mc_t-P_ana)*1e4:+.2f} bps")

# Exposure profile skeleton — will be consumed by CVA in QRE-53
dates = np.arange(1.0, 11.0, 1.0)   # annual grid 1Y–10Y

# A simple receiver swap MTM proxy: positive when rates fall below the fixed leg
swap_fixed_rate = r0_market    # at-the-money at inception (percent)
def swap_mtm(paths: np.ndarray, t: float) -> np.ndarray:
    """Simplified swap MTM at time t: proportional to r(t) - fixed_rate.
    Sign convention: positive for the fixed receiver when rates are low."""
    t_idx = int(round(t / sim.dt))
    t_idx = max(0, min(t_idx, sim.n_steps))
    # Receiver swap: long fixed, short floating → MTM > 0 when r falls below fixed
    return (swap_fixed_rate - paths[:, t_idx]) * (sim.T - t)

profile = sim.exposure_profile(swap_mtm, dates)
print(f"\nExposure profile (simplified receiver swap proxy)")
print(f"  {'Date':>6}  {'EE':>8}  {'NEE':>8}  {'PFE95':>8}")
for i, t in enumerate(profile['dates']):
    print(f"  {t:>6.1f}Y  {profile['EE'][i]:>8.4f}  {profile['NEE'][i]:>8.4f}  "
          f"{profile['PFE'][i]:>8.4f}")
print(f"  EPE (time-avg EE): {profile['EPE']:.4f}")


---
## Summary

| Step | What we did | Key formula |
|---|---|---|
| **Theory** | Derived the Vasicek SDE, analytical solution, and stationary distribution | $r(t) \sim \mathcal{N}(\mu(t), v^2(t))$ |
| **Bond pricing** | Derived $\beta(T)$ and $\alpha(T)$ from the bond pricing PDE by separation of variables | $P(0,T) = e^{\alpha(T) - \beta(T)r_0}$ |
| **Yield curve shapes** | Showed how $\kappa$ controls convergence speed, $\sigma$ suppresses the long end, $r_0$ controls level | $R_\infty = \theta - \sigma^2/(2\kappa^2)$ |
| **Calibration** | Fitted $(\kappa, \theta, \sigma)$ to OIS zero yields via weighted least-squares | $\min \sum w_i (R^{\text{model}}_i - R^{\text{OIS}}_i)^2$ |
| **1-factor limit** | Residuals at 2Y–10Y show the model cannot fit curvature; Hull-White resolves this | HW: $dr = (\theta(t) - \kappa r)\,dt + \sigma\,dW$ |
| **MC ZCB pricing** | Verified $E^{\mathbb{Q}}[e^{-\int_0^T r\,ds}] \to P^{\text{Vasicek}}(0,T)$ as $N \to \infty$ | $O(1/\sqrt{N})$ convergence; antithetic ~3–5× speedup |
| **Exposure profile** | Used `MCSimulator.exposure_profile()` to produce EE/NEE/PFE — the XVA inputs | $\text{EE}(t) = E^{\mathbb{Q}}[\max(V(t), 0)]$ |

**Next:** [QRE-44 — Hull-White](06_hw_model.ipynb) — time-dependent drift $\theta(t)$ calibrated to the full OIS curve, swaption pricing, and the connection to the `HullWhiteProcess` class already implemented in `src/quant_risk/models/rates.py`.
